# Exp6 Router Ablation — Train Qwen2.5-1.5B-Instruct

**Size**: qwen25_15b  
**Base model**: `unsloth/Qwen2.5-1.5B-Instruct`  
**Output repo**: `daredevil467/hanoi-router-qwen25-15b`  
**Data**: [`daredevil467/hanoi-weather-router-data`](https://huggingface.co/datasets/daredevil467/hanoi-weather-router-data) (train 3366, val 373)  
**Effective batch**: 8 × 4 = 32

Run on Colab A100 (premium GPU). Before running, set `HF_TOKEN` in Colab Secrets (🔑 panel).

In [ ]:
%%capture
!pip install -U "unsloth[colab-new]" "trl>=0.17,<0.19" "huggingface_hub>=0.26" datasets


In [ ]:
import os, torch, subprocess
from google.colab import userdata
from huggingface_hub import login

# HF login from Colab Secrets (🔑 icon in left panel)
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Set HF_TOKEN in Colab Secrets before running'
login(token=HF_TOKEN)

gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader']).decode().strip()
print(f'GPU: {gpu}')
print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
assert torch.cuda.is_bf16_supported(), 'A100 (or better) required for bf16 training'


In [ ]:
# ═══════════════════════ CONFIG (Exp6 shared recipe) ═══════════════════════
BASE_MODEL     = 'unsloth/Qwen2.5-1.5B-Instruct'
HF_REPO        = 'daredevil467/hanoi-router-qwen25-15b'
HF_DATA_REPO   = 'daredevil467/hanoi-weather-router-data'

MAX_SEQ_LENGTH = 1024
LORA_R         = 32          # Identical to v4 recipe
LORA_ALPHA     = 64
LORA_DROPOUT   = 0.1
EPOCHS         = 10
BATCH_SIZE     = 8
GRAD_ACCUM     = 4
LR             = 2e-4
WARMUP_RATIO   = 0.06
EVAL_STEPS     = 50
OUTPUT_DIR     = '/content/outputs'

print(f'Model      : {BASE_MODEL}')
print(f'LoRA       : r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}')
print(f'Batch      : {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM} effective')
print(f'Epochs     : {EPOCHS}')
print(f'LR         : {LR}')
print(f'HF target  : {HF_REPO}')


In [ ]:
# ═══════════════════════ DATA DOWNLOAD from HF ═══════════════════════
from huggingface_hub import hf_hub_download
import json as _json

train_file = hf_hub_download(repo_id=HF_DATA_REPO, filename='multitask_train_v5_clean.jsonl', repo_type='dataset')
val_file   = hf_hub_download(repo_id=HF_DATA_REPO, filename='multitask_val_v5_clean.jsonl',   repo_type='dataset')
prompt_file= hf_hub_download(repo_id=HF_DATA_REPO, filename='system_prompt.txt',              repo_type='dataset')

with open(train_file, encoding='utf-8') as f:
    train_raw = [_json.loads(l) for l in f]
with open(val_file, encoding='utf-8') as f:
    val_raw = [_json.loads(l) for l in f]
with open(prompt_file, encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

print(f'Train: {len(train_raw)} samples')
print(f'Val:   {len(val_raw)} samples')
print(f'System prompt: {len(SYSTEM_PROMPT)} chars')
assert len(train_raw) == 3366, f'Expected 3366 train rows, got {len(train_raw)}'
assert len(val_raw)   == 373,  f'Expected 373 val rows, got {len(val_raw)}'


In [ ]:
# ═══════════════════════ LOAD BASE MODEL ═══════════════════════
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = BASE_MODEL,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = torch.bfloat16,
    load_in_4bit   = False,
)
print(f'Model loaded: {BASE_MODEL}')
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# ═══════════════════════ APPLY LoRA ═══════════════════════
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)')


In [ ]:
# ═══════════════════════ TOKENIZER SETUP (manual ChatML) ═══════════════════════
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'right'

# Verify ChatML special tokens exist (Qwen2/3 family)
im_start_ids = tokenizer.encode('<|im_start|>', add_special_tokens=False)
im_end_ids   = tokenizer.encode('<|im_end|>',   add_special_tokens=False)
print(f'<|im_start|> token ids: {im_start_ids}')
print(f'<|im_end|> token ids:   {im_end_ids}')
assert len(im_start_ids) == 1 and len(im_end_ids) == 1, 'ChatML special tokens missing'


In [ ]:
# ═══════════════════════ FORMAT RECORDS (ChatML, identical to v4) ═══════════════════════
import json
from datasets import Dataset

IM_START = '<|im_start|>'
IM_END   = '<|im_end|>'
NL       = chr(10)

def format_record(rec):
    user_msg = str(rec.get('input', '')).strip()
    ctx = rec.get('context')
    if ctx:
        ctx_str  = json.dumps(ctx, ensure_ascii=False, separators=(',', ':'))
        user_msg = '[CONTEXT: ' + ctx_str + ']' + NL + user_msg
    out = rec.get('output', {})
    if isinstance(out, str):
        out = json.loads(out)
    output_dict = {
        'intent':     out['intent'],
        'scope':      out['scope'],
        'confidence': round(float(out.get('confidence', 0.9)), 2),
    }
    rw = out.get('rewritten_query')
    if rw and str(rw).strip():
        output_dict['rewritten_query'] = str(rw).strip()
    text  = IM_START + 'system'    + NL + SYSTEM_PROMPT + IM_END + NL
    text += IM_START + 'user'      + NL + user_msg      + IM_END + NL
    text += IM_START + 'assistant' + NL + json.dumps(output_dict, ensure_ascii=False) + IM_END + NL
    return text

train_texts = [format_record(r) for r in train_raw]
val_texts   = [format_record(r) for r in val_raw]

lengths = [len(tokenizer.encode(t)) for t in train_texts[:200]]
print(f'Avg tokens: {sum(lengths)/len(lengths):.0f}, Max: {max(lengths)}, Over {MAX_SEQ_LENGTH}: {sum(1 for l in lengths if l > MAX_SEQ_LENGTH)}')

raw_train = Dataset.from_dict({'text': train_texts})
raw_val   = Dataset.from_dict({'text': val_texts})
print(f'Train: {len(raw_train)}, Val: {len(raw_val)}')


In [ ]:
# ═══════════════════════ COMPLETION-ONLY MASKING (v3 key fix) ═══════════════════════
# Mask everything up to '<|im_start|>assistant\n'. Loss only on the JSON response.
ASSISTANT_MARKER = '<|im_start|>assistant' + chr(10)
assistant_ids = tokenizer.encode(ASSISTANT_MARKER, add_special_tokens=False)
print(f'Assistant marker ids: {assistant_ids}')

def tokenize_with_masking(examples):
    enc = tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding=False,
        return_tensors=None,
    )
    all_labels = []
    for input_ids in enc['input_ids']:
        labels = [-100] * len(input_ids)
        marker_len = len(assistant_ids)
        for i in range(len(input_ids) - marker_len + 1):
            if input_ids[i:i+marker_len] == assistant_ids:
                start = i + marker_len
                for j in range(start, len(input_ids)):
                    labels[j] = input_ids[j]
                break
        else:
            labels = list(input_ids)  # fallback (should not happen)
        all_labels.append(labels)
    enc['labels'] = all_labels
    return enc

train_dataset = raw_train.map(tokenize_with_masking, batched=True, remove_columns=['text'])
val_dataset   = raw_val.map(tokenize_with_masking, batched=True, remove_columns=['text'])

# Sanity check
sl = train_dataset[0]['labels']
masked = sum(1 for x in sl if x == -100)
print(f'Sample: {len(sl)} total, {masked} masked ({masked/len(sl)*100:.1f}%), {len(sl)-masked} active')


In [ ]:
# ═══════════════════════ TRAINER SETUP ═══════════════════════
from transformers import DataCollatorForSeq2Seq, EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

data_collator = DataCollatorForSeq2Seq(
    tokenizer          = tokenizer,
    model              = model,
    label_pad_token_id = -100,
    pad_to_multiple_of = 8,
)

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    warmup_ratio                = WARMUP_RATIO,
    lr_scheduler_type           = 'cosine',
    logging_steps               = 10,
    eval_strategy               = 'steps',
    eval_steps                  = EVAL_STEPS,
    save_strategy               = 'steps',
    save_steps                  = EVAL_STEPS,
    save_total_limit            = 2,
    metric_for_best_model       = 'eval_loss',
    load_best_model_at_end      = True,
    bf16                        = True,
    fp16                        = False,
    optim                       = 'adamw_torch_fused',
    weight_decay                = 0.01,
    max_grad_norm               = 1.0,
    max_seq_length              = MAX_SEQ_LENGTH,
    report_to                   = 'none',
)

early_stop = EarlyStoppingCallback(early_stopping_patience=5, early_stopping_threshold=0.001)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    args             = training_args,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    data_collator    = data_collator,
    packing          = False,
    callbacks        = [early_stop],
)
print('Trainer ready')


In [ ]:
# ═══════════════════════ TRAIN ═══════════════════════
trainer_stats = trainer.train()
print(trainer_stats)


In [ ]:
# ═══════════════════════ SAVE ADAPTER + MERGE + PUSH to HF ═══════════════════════
# 1. Save LoRA adapter locally
adapter_dir = '/content/outputs/lora_adapter'
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f'Adapter saved to {adapter_dir}')

# 2. Free disk (checkpoints) before merge
import shutil, gc
from pathlib import Path
out = Path(OUTPUT_DIR)
for ckpt in sorted(out.glob('checkpoint-*')):
    print(f'Deleting {ckpt.name}...')
    shutil.rmtree(ckpt)
torch.cuda.empty_cache()
gc.collect()

# 3. Push merged model (16-bit) to HF Hub — used by eval notebook 06
print(f'Pushing merged model to HF: {HF_REPO}')
model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method='merged_16bit',
    token=HF_TOKEN,
)
print(f'Pushed: https://huggingface.co/{HF_REPO}')


In [ ]:
# ═══════════════════════ (OPTIONAL) Push GGUF Q4_K_M for Ollama production ═══════════════════════
# Run this cell only for the size you want to deploy via Ollama locally.
# GGUF export takes ~5-15 min depending on model size; safe to skip for ablation-only runs.

GGUF_HF_REPO = HF_REPO + '-gguf'
print(f'Pushing GGUF Q4_K_M to {GGUF_HF_REPO}...')
model.push_to_hub_gguf(
    GGUF_HF_REPO,
    tokenizer,
    quantization_method='q4_k_m',
    token=HF_TOKEN,
)
print(f'Pushed GGUF: https://huggingface.co/{GGUF_HF_REPO}')
